In [ ]:
#@title Installations
%%capture

!pip install nilearn
!pip install decord

In [ ]:
# prompt: load alexnet

import torch
import torchvision
import glob
import numpy as np
import urllib
import torch
import cv2
import argparse
import time
import random
import matplotlib.pyplot as plt
import nibabel as nib
import pickle
from nilearn import plotting
from tqdm import tqdm
from torchvision import transforms as trn
import os
from PIL import Image
from sklearn.preprocessing import StandardScaler
from torch.autograd import Variable as V
from sklearn.decomposition import PCA, IncrementalPCA
import torch.nn as nn
import torch.utils.model_zoo as model_zoo
import torch.nn.functional as F
from nilearn import datasets
from nilearn import surface
from decord import VideoReader
from decord import cpu

#alexnet_model = torchvision.models.alexnet(pretrained=True)

In [ ]:
#@title Enter the download link and run the cell
download_link = "'https://www.dropbox.com/s/agxyxntrbwko7t1/participants_data.zip?dl=1'" #@param {type:"string"}
os.environ["download_link"] = download_link
!echo $download_link
!wget -O participants_data.zip -c $download_link
!unzip participants_data.zip
!wget -O example.nii -c https://github.com/Neural-Dynamics-of-Visual-Cognition-FUB/Algonauts2021_devkit/raw/main/example.nii
!wget -c https://raw.githubusercontent.com/Neural-Dynamics-of-Visual-Cognition-FUB/Algonauts2021_devkit/main/class_names_ImageNet.txt

In [ ]:
vid_id = 2 #@param {type: "integer"}
video_dir = '/content/AlgonautsVideos268_All_30fpsmax'

########### Video display code #################################################
video_list = glob.glob(video_dir + '/*.mp4')
video_list.sort()


#visualize_brain_activity(video_dir,fmri_train_all,vid_id)
from IPython.display import HTML
from base64 import b64encode

mp4 = open(video_list[vid_id],'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=400 controls>
    <source src="%s" type="video/mp4">
</video>
""" % data_url)

In [3]:
import os
import numpy as np
from PIL import Image
from decord import VideoReader, cpu
import torch
from torchvision import transforms as trn
from tqdm import tqdm

# --- Preprocessing transforms ---
resize_normalize = trn.Compose([
    trn.Resize((224, 224)),
    trn.ToTensor(),
    trn.Normalize(mean=[0.485, 0.456, 0.406],
                  std=[0.229, 0.224, 0.225])
])

# --- Sample uniformly from a frame range ---
def sample_video_from_mp4(file, num_frames=16, frame_range=(0, None)):
    """
    Sample uniformly from a specific frame range in the video.

    Parameters:
        file (str): Path to .mp4 video.
        num_frames (int): Number of frames to sample.
        frame_range (tuple): (start_frame, end_frame) for subsetting.

    Returns:
        frames (List[PIL.Image]): Sampled RGB frames.
    """
    vr = VideoReader(file, ctx=cpu(0))
    total_frames = len(vr)

    start, end = frame_range
    if end is None or end > total_frames:
        end = total_frames
    if start < 0 or start >= end:
        raise ValueError(f"Invalid frame_range: ({start}, {end})")

    available = end - start
    indices = np.linspace(start, end - 1, min(num_frames, available), dtype=int)
    frames = [Image.fromarray(vr[i].asnumpy()).convert("RGB") for i in indices]
    return frames

# --- Preprocess individual frames ---
def preprocess_frames(frames):
    """
    Apply resizing and normalization to frames and stack into tensor.

    Returns:
        Tensor of shape (C, T, H, W)
    """
    tensors = [resize_normalize(img) for img in frames]
    video_tensor = torch.stack(tensors, dim=1)  # (C, T, H, W)
    return video_tensor

# --- Apply hemodynamic delay shift ---
def apply_hemodynamic_delay(video_tensor, delay_frames=2):
    """
    Shift frames to simulate hemodynamic delay.

    Returns:
        Tensor of same shape with delayed start.
    """
    C, T, H, W = video_tensor.shape
    delayed = torch.zeros_like(video_tensor)
    if T > delay_frames:
        delayed[:, delay_frames:] = video_tensor[:, :-delay_frames]
    return delayed

def process_videos_to_tensor(
    video_list,
    num_frames=16,
    delay_frames=2,
    frame_range=(0, None),
    video_range=(0, None)
):
    """
    Process a subset of videos into a single tensor of shape (N, C, T, H, W).

    Parameters:
        video_list (List[str]): Paths to video files.
        num_frames (int): Frames to sample per video.
        delay_frames (int): Shift for hemodynamic delay.
        frame_range (tuple): (start, end) frame indices to sample within each video.
        video_range (tuple): (start, end) indices of videos to process from the list.

    Returns:
        Tensor of shape (N, C, T, H, W)
    """
    start_vid, end_vid = video_range
    if end_vid is None or end_vid > len(video_list):
        end_vid = len(video_list)
    if start_vid < 0 or start_vid >= end_vid:
        raise ValueError(f"Invalid video_range: ({start_vid}, {end_vid})")

    selected_videos = video_list[start_vid:end_vid]
    processed_videos = []

    for video_file in tqdm(selected_videos, desc=f"Processing videos {start_vid}–{end_vid}"):
        try:
            frames = sample_video_from_mp4(video_file, num_frames=num_frames, frame_range=frame_range)

            video_tensor = preprocess_frames(frames)  # (C, T, H, W)
            #print(video_tensor)
            #video_tensor = apply_hemodynamic_delay(video_tensor, delay_frames=delay_frames)
            #print(video_tensor)
           # print(processed_videos)
            processed_videos.append(video_tensor)
            #print(processed_videos)

        except Exception as e:
            print(f"Skipping {video_file} due to error: {e}")

    if not processed_videos:
        raise RuntimeError("No videos were successfully processed.")

    return torch.stack(processed_videos, dim=0)  # (N, C, T, H, W)

video_tensors = process_videos_to_tensor(video_list, num_frames=15, delay_frames=2,frame_range=(0,100),video_range=(0,100))


ModuleNotFoundError: No module named 'decord'

In [ ]:

plt.imshow(video_tensors[0,2,13])

In [ ]:
import os
import numpy as np
from PIL import Image
from decord import VideoReader, cpu
import torch
from torchvision import transforms as trn
from tqdm import tqdm

def sample_video_from_mp4(file, num_frames=16):
    """Uniformly sample frames from a video and convert to RGB PIL images."""
    vr = VideoReader(file, ctx=cpu(0))
    total_frames = len(vr)
    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    images = [Image.fromarray(vr[i].asnumpy()).convert("RGB") for i in indices]
    return images

# Preprocessing: Resize, ToTensor, Normalize
resize_normalize = trn.Compose([
    trn.Resize((224, 224)),
    trn.ToTensor(),
    trn.Normalize(mean=[0.485, 0.456, 0.406],
                  std=[0.229, 0.224, 0.225])
])

def preprocess_frames(frames):
    """Apply preprocessing and return a tensor of shape (C, T, H, W)."""
    tensors = [resize_normalize(img) for img in frames]
    video_tensor = torch.stack(tensors, dim=1)  # (C, T, H, W)
    return video_tensor

def apply_hemodynamic_delay(video_tensor, delay_frames=2):
    """Shift frames to account for hemodynamic response delay."""
    C, T, H, W = video_tensor.shape
    delayed_tensor = torch.zeros_like(video_tensor)
    delayed_tensor[:, delay_frames:] = video_tensor[:, :-delay_frames]
    return delayed_tensor

def process_videos_to_tensor(video_list, vids_list,num_frames=16, delay_frames=2):
    """Preprocess videos and return a single tensor of shape (N, C, T, H, W)."""
    processed_videos = []

    for video_file in tqdm(video_list[vids_list[0]:vids_list[1]]):
        frames = sample_video_from_mp4(video_file, num_frames=num_frames)
        video_tensor = preprocess_frames(frames)              # (C, T, H, W)
        video_tensor = apply_hemodynamic_delay(video_tensor, delay_frames=delay_frames)
        processed_videos.append(video_tensor)

    # Stack all videos into a single tensor: (N, C, T, H, W)
    all_videos_tensor = torch.stack(processed_videos, dim=0)
    return all_videos_tensor

video_tensors = process_videos_to_tensor(video_list,[0:100], num_frames=16, delay_frames=2)

# Example: Access the preprocessed tensor for a specific video
#tensor = video_tensors[1]  # tensor shape: (C, T, H, W)

SyntaxError: invalid syntax (ipython-input-7-3272471637.py, line 52)

In [ ]:
video_tensors[0]

In [ ]:

    """This function generates Alexnet features and save them in a specified directory.
    Parameters
    ----------
    model :
        pytorch model : alexnet.
    video_list : list
        the list contains path to all videos.
    activations_dir : str
        save path for extracted features.
    """
    def sample_video_from_mp4(file, num_frames=16):
      """This function takes a mp4 video file as input and returns
      a list of uniformly sampled frames (PIL Image).
      Parameters
      ----------
      file : str
          path to mp4 video file
      num_frames : int
          how many frames to select using uniform frame sampling.
      Returns
      -------
      images: list of PIL Images
      num_frames: int
          number of frames extracted
      """
      images = list()
      vr = VideoReader(file, ctx=cpu(0))
      total_frames = len(vr)
      indices = np.linspace(0,total_frames-1,num_frames,dtype=int)
      for seg_ind in indices:
          images.append(Image.fromarray(vr[seg_ind].asnumpy()))

      return images,num_frames

    resize_normalize = trn.Compose([
            trn.Resize((224,224)),
            trn.ToTensor(),
            trn.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

    for vi,video_file in enumerate(tqdm(video_list)):
        vid,num_frames = sample_video_from_mp4(video_file)
        video_file_name = os.path.split(video_file)[-1].split(".")[0]
        activations = []
        for frame,img in enumerate(vid):
            input_img = V(resize_normalize(img).unsqueeze(0))
            if vi==0:
              print(input_img.shape)
            #if torch.cuda.is_available():
                #input_img=input_img.cuda()
           # x = model.forward(input_img)
            #for i,feat in enumerate(x):
                #if frame==0:
                    #if i==7:
                        #print("\nTop-5 Predictions for the video id: ", video_file_name)
                        #print_alexnet_predictions(feat)
                   # activations.append(feat.data.cpu().numpy().ravel())
                #else:
                    #activations[i] =  activations[i] + feat.data.cpu().numpy().ravel()
        #for layer in range(len(activations)):
            #save_path = os.path.join(activations_dir, video_file_name+"_"+"layer" + "_" + str(layer+1) + ".npy")
            #avg_layer_activation = activations[layer]/float(num_frames)
            #np.save(save_path,avg_layer_activation)

- Extract frames at consistent ~5 fps rate
- Convert to rgb and resize to 224x224
- Apply ImageNet normalization to each frame, convert frames to pytorch tensors and arrange them in the following format: channels, time, spatial dimensions
- Use small batch size
- Account for hemodynamic response delay.  

In [ ]:
## Architecture

class VideoClipToMRI(nn.Module):
    def __init__(self, num_voxels):
        super().__init__()

        # Pretrained spatial encoder
        alexnet = torchvision.models.alexnet(pretrained=True)
        self.spatial_encoder = alexnet.features
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))

        # Temporal modeling
        self.temporal_encoder = nn.LSTM(
            input_size=9216,  # ResNet50 output size
            hidden_size=512,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        # MRI voxel prediction
        self.voxel_predictor = nn.Sequential(
            nn.Linear(1024, 2048),  # 1024 from bidirectional LSTM
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(2048, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, num_voxels)
        )

    def forward(self, video_clips):
        batch_size, channels, time, height, width = video_clips.size()

        # Extract spatial features per frame
        spatial_features = []
        for t in range(time):
            frame = video_clips[:, :, t, :, :]  # [batch, 3, H, W]
            features = self.spatial_encoder(frame)  # [batch, 2048, 1, 1]
            features = features.view(batch_size, -1)  # [batch, 2048]
            spatial_features.append(features)

        # Stack temporal sequence
        sequence = torch.stack(spatial_features, dim=1)  # [batch, time, 2048]

        # Temporal modeling
        lstm_out, (hidden, cell) = self.temporal_encoder(sequence)

        # Use final hidden state for prediction
        final_features = torch.cat([hidden[-2], hidden[-1]], dim=1)  # Concat both directions

        # Predict voxel activations
        voxel_predictions = self.voxel_predictor(final_features)

        return voxel_predictions

In [ ]:
## Data Preprocessing

class VideoMRIDataset(Dataset):
    def __init__(self, video_paths, mri_data, transform=None):
        self.video_paths = video_paths
        self.mri_data = mri_data
        self.transform = transform

    def __getitem__(self, idx):
        # Load video clip
        video = self.load_video(self.video_paths[idx])

        # Load corresponding MRI data
        mri_voxels = self.mri_data[idx]

        if self.transform:
            video = self.transform(video)

        return video, mri_voxels

    def load_video(self, video_path):
        # Load video frames
        frames = []
        cap = cv2.VideoCapture(video_path)

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (224, 224))
            frames.append(frame)

        cap.release()

        # Convert to tensor [time, channels, height, width]
        video_tensor = torch.tensor(np.array(frames)).float() / 255.0
        video_tensor = video_tensor.permute(0, 3, 1, 2)  # [T, C, H, W]

        return video_tensor.permute(1, 0, 2, 3)  # [C, T, H, W]

Preserve

All convolutional layers (conv1-conv5)
Spatial feature extraction architecture
Pretrained weights for visual processing

Replace

Classification head → Regression head
Single image input → Video sequence input
Categorical output → Continuous voxel predictions

Add

Temporal aggregation mechanism (LSTM/attention)
Frame-by-frame processing loop
Adaptive pooling for consistent feature sizes

Implementation Approach
Input Flow
Video sequences → Frame-wise spatial processing → Temporal feature aggregation → Voxel prediction
Key Design Decisions

Choose temporal processing method (LSTM vs attention vs averaging)
Determine which conv layers to extract features from
Decide on single vs multi-head prediction architecture
Select appropriate loss function for regression task

~ Use torchvision.models.alexnet